In [182]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Running on {len(gpus)} GPU(s).")
    except RuntimeError as e:
        print(e)

In [183]:
# Load data
# Kaggle notebooks read data from the ../input/ directory
df_train = pd.read_csv("dataset/train.csv", index_col="id")
df_test = pd.read_csv("dataset/train.csv", index_col="id")

X = df_train.copy()
y = X.pop('accident_risk')

In [184]:
# ---------------------------------------------------------------------------
# FEATURE ENGINEERING
# ---------------------------------------------------------------------------

# 1. Ordinal Encoding for features with a natural order
lighting_map = {'night': 0, 'dim': 1, 'daylight': 2}
time_map = {'morning': 0, 'afternoon': 1, 'evening': 2}

X['lighting'] = X['lighting'].map(lighting_map)
X['time_of_day'] = X['time_of_day'].map(time_map)
df_test['lighting'] = df_test['lighting'].map(lighting_map)
df_test['time_of_day'] = df_test['time_of_day'].map(time_map)


# 2. One-Hot Encoding for the remaining categorical and boolean features
bool_cols = [col for col in X.columns if X[col].dtype == 'bool']
remaining_object_cols = ['road_type', 'weather']

X = pd.get_dummies(X, columns=remaining_object_cols + bool_cols, dummy_na=False)
df_test = pd.get_dummies(df_test, columns=remaining_object_cols + bool_cols, dummy_na=False)


# 3. Align columns to ensure train and test sets have the same features
X, df_test = X.align(df_test, join='left', axis=1, fill_value=0)


In [ ]:
def create_features(df):
    """Advanced feature engineering"""
    df = df.copy()
    
    # 1. Polynomial features for key numerical variables
    df['curvature_squared'] = df['curvature'] ** 2
    df['curvature_cubed'] = df['curvature'] ** 3
    df['speed_squared'] = df['speed_limit'] ** 2
    
    # 2. Binned features
    df['curvature_bin'] = pd.cut(df['curvature'], bins=[0, 0.3, 0.6, 1.0], labels=[0, 1, 2])
    df['speed_category'] = pd.cut(df['speed_limit'], bins=[0, 30, 50, 100], labels=[0, 1, 2])
    
    # 3. Interaction features
    df['speed_curvature'] = df['speed_limit'] * df['curvature']
    df['lanes_curvature'] = df['num_lanes'] * df['curvature']
    df['speed_lanes'] = df['speed_limit'] * df['num_lanes']
    df['accidents_curvature'] = df['num_reported_accidents'] * df['curvature']
    df['accidents_speed'] = df['num_reported_accidents'] * df['speed_limit']
    
    # 4. Risk score combinations
    df['high_risk_combo'] = ((df['curvature'] > 0.5) & (df['speed_limit'] >= 60)).astype(int)
    df['weather_lighting_risk'] = ((df['weather'] == 'foggy') | (df['weather'] == 'rainy')) & \
                                   ((df['lighting'] == 'dim') | (df['lighting'] == 'night'))
    df['weather_lighting_risk'] = df['weather_lighting_risk'].astype(int)
    
    # 5. Categorical aggregations (target encoding will be done in CV)
    df['is_night'] = (df['lighting'] == 'night').astype(int)
    df['is_bad_weather'] = df['weather'].isin(['foggy', 'rainy']).astype(int)
    df['is_highway'] = (df['road_type'] == 'highway').astype(int)
    df['is_urban'] = (df['road_type'] == 'urban').astype(int)
    
    # 6. Time-based features
    df['is_peak_time'] = df['time_of_day'].isin(['morning', 'evening']).astype(int)
    df['is_weekend'] = df['holiday'].astype(int)  # Using holiday as proxy
    
    # 7. Safety features
    df['safety_score'] = df['road_signs_present'].astype(int) * 2 + \
                         (df['lighting'] == 'daylight').astype(int) + \
                         (df['weather'] == 'clear').astype(int)
    
    df['danger_score'] = (df['curvature'] > 0.6).astype(int) + \
                         (df['speed_limit'] >= 60).astype(int) + \
                         df['is_bad_weather'] + df['is_night'] + \
                         (df['num_reported_accidents'] >= 2).astype(int)
    
    # 8. Ratio features
    df['accidents_per_lane'] = df['num_reported_accidents'] / (df['num_lanes'] + 1)
    df['risk_intensity'] = df['curvature'] * df['speed_limit'] / 50
    
    return df

In [ ]:
# ---------------------------------------------------------------------------
# NEURAL NETWORK STACKING MODEL (with Efficient Data Pipeline)
# ---------------------------------------------------------------------------

# Define the Neural Network Architecture
def get_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=[X.shape[1]]), # Recommended way to define input
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Setup cross-validation
N_SPLITS = 10
SEED = [0, 1, 2, 42]
BATCH_SIZE = 2048 # You can often increase batch size with a better pipeline

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Initialize arrays for predictions
oof_preds = np.zeros((len(X),))
test_preds = np.zeros((len(df_test),))

# Scale data before the loop
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(df_test)

print("Starting Neural Network training...")
# Training loop
for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled, y)):
    print(f"===== Fold {fold+1} =====")
    X_train_scaled, X_val_scaled = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # ✅ FIX: Create efficient tf.data pipelines
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train_scaled, y_train))
    train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val_scaled, y_val))
    val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    val_preds_sum = np.zeros(len(val_idx))
    test_preds_sum = np.zeros(len(df_test))

    for s in SEED:
        tf.random.set_seed(s)
        model = get_model()

        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )

        model.fit(train_dataset,
                  validation_data=val_dataset,
                  epochs=200,
                  callbacks=[early_stopping],
                  verbose=0
                 )

        val_preds_sum += model.predict(X_val_scaled).flatten()
        test_preds_sum += model.predict(X_test_scaled).flatten()

    # Average predictions over seeds for this fold
    oof_preds[val_idx] = val_preds_sum / len(SEED)
    test_preds += (test_preds_sum / len(SEED)) / N_SPLITS # Average test preds over folds

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"Fold {fold+1} RMSE: {fold_rmse:.5f}")

# Evaluate the overall model performance
overall_oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\nOverall OOF RMSE for Neural Network: {overall_oof_rmse:.5f}")


In [ ]:
# ---------------------------------------------------------------------------
# CREATE SUBMISSION FILE
# ---------------------------------------------------------------------------
print("\nCreating submission file...")
submission = pd.DataFrame({'id': df_test.index, 'accident_risk': test_preds})
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")